# 🏥 환자 맞춤형 의료 안내 서비스
**Mini AIFFELthon — AI-Hub 전문 의학지식 데이터 활용**

```
[입력: 소견서 텍스트 + 거주 지역]
  → Step 1. RAG 검색 (TF-IDF, 국문 원천데이터 ~2만 청크)
  → Step 2. 쉬운 말 설명 생성 (LLM + Few-shot)
  → Step 3. 진료과 · 중증도 추출 (LLM)
  → Step 4. 권역별 병원 매칭 (Track A/B)
  → Step 5. 최종 리포트 합성 (LLM)
  → Step 6. LLM-as-Judge 품질 검증
  → Step 7. Gradio UI 실행
```

> ⚠️ **면책 고지**: 본 서비스는 의료 정보 제공 목적의 AI 보조 도구입니다. 진단·처방을 대체하지 않으며, 최종 의료 판단은 반드시 전문의에게 받으시기 바랍니다.

---
## 0. 패키지 설치 및 환경 설정

In [ ]:
# 필수 패키지 설치
!pip install gradio scikit-learn numpy anthropic -q

In [ ]:
import os
import json
import glob
import pickle
import random
import re
from collections import defaultdict
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import anthropic

# ── API 키 설정 ──────────────────────────────────────────────
# 방법 1: 직접 입력 (권장하지 않음, 공유 금지)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

# 방법 2: 환경변수로 미리 설정 후 확인
api_key = os.environ.get('ANTHROPIC_API_KEY', '')
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY 환경변수를 먼저 설정하세요.\n"
                     "터미널: export ANTHROPIC_API_KEY=sk-ant-...")

client = anthropic.Anthropic(api_key=api_key)
print("✅ Anthropic 클라이언트 초기화 완료")

In [ ]:
# ── 데이터 경로 설정 ─────────────────────────────────────────
# 이 노트북 파일과 같은 폴더 기준으로 자동 설정
BASE_DIR = os.getcwd()

TS_DIR = os.path.join(BASE_DIR, 'ts_data')  # 원천데이터
TL_DIR = os.path.join(BASE_DIR, 'tl_data')  # Training 라벨링
VL_DIR = os.path.join(BASE_DIR, 'vl_data')  # Validation 라벨링

for d, name in [(TS_DIR,'ts_data'), (TL_DIR,'tl_data'), (VL_DIR,'vl_data')]:
    status = '✅' if os.path.isdir(d) else '❌ 없음'
    print(f"{status} {name}/")

# 공통 상수
DOMAIN_MAP = {
    1:'외과', 2:'예방의학', 3:'정신건강의학과', 4:'신경과신경외과',
    5:'피부과', 6:'안과', 7:'이비인후과', 8:'비뇨의학과',
    9:'방사선종양학과', 10:'병리과', 11:'마취통증의학과', 12:'의료법규', 13:'기타'
}

---
## 1. RAG 인덱스 구축
국문 원천데이터(TS) → TF-IDF 벡터화 → `tfidf_index.pkl`

> 최초 1회만 실행. 이후에는 **셀 1-2에서 pkl 로드**로 건너뜁니다.

In [ ]:
# ── 셀 1-1: 압축 해제 + 인덱스 구축 (최초 1회) ─────────────────
# pkl 파일이 이미 있으면 다음 셀(1-2)로 건너뛰세요.

import zipfile, os, glob, json, pickle
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

INDEX_PATH = os.path.join(BASE_DIR, 'tfidf_index.pkl')

if os.path.exists(INDEX_PATH):
    print("ℹ️  tfidf_index.pkl 이미 존재. 다음 셀(1-2)로 건너뛰세요.")
else:
    # ── Step 1: 국문 원천데이터 압축 해제 ──────────────────────
    TS_TARGETS = [
        'TS_국문_의학_교과서',
        'TS_국문_학술_논문_및_저널',
        'TS_국문_학회_가이드라인',
        'TS_국문_온라인_의료_정보_제공_사이트',
    ]
    TS_EXTRACT_DIR = os.path.join(BASE_DIR, 'ts_data')
    os.makedirs(TS_EXTRACT_DIR, exist_ok=True)

    for name in TS_TARGETS:
        zip_path = os.path.join(TS_DIR, f'{name}.zip.part0')
        out_dir  = os.path.join(TS_EXTRACT_DIR, name)
        if os.path.isdir(out_dir) and len(glob.glob(f'{out_dir}/**/*.json', recursive=True)) > 0:
            print(f"  ⏭️  {name} 이미 해제됨")
            continue
        os.makedirs(out_dir, exist_ok=True)
        try:
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_dir)
            count = len(glob.glob(f'{out_dir}/**/*.json', recursive=True))
            print(f"  ✅ {name}: {count}개 JSON")
        except Exception as e:
            print(f"  ❌ {name}: {e}")

    # ── Step 2: TF-IDF 인덱스 구축 ─────────────────────────────
    CATS = {
        'TS_국문_의학_교과서': 159,
        'TS_국문_학술_논문_및_저널': 970,
        'TS_국문_학회_가이드라인': 1298,
        'TS_국문_온라인_의료_정보_제공_사이트': 3000,
    }

    print("\n문서 청킹 중...")
    docs = []
    for cat, limit in CATS.items():
        cat_dir = os.path.join(TS_EXTRACT_DIR, cat)
        files   = glob.glob(f'{cat_dir}/**/*.json', recursive=True)
        loaded  = 0
        for fp in files[:limit]:
            with open(fp, 'r', encoding='utf-8-sig') as f:
                try:
                    d = json.load(f)
                    content = d.get('content', '').strip()
                    if len(content) < 100:
                        continue
                    for i in range(0, min(len(content), 2000), 400):
                        chunk = content[i:i+500]
                        if len(chunk) > 80:
                            docs.append({
                                'text': chunk, 'cat': cat,
                                'source': str(d.get('source_spec', '')),
                                'domain': d.get('domain'),
                            })
                            loaded += 1
                except:
                    pass
        print(f"  {cat}: {loaded}개 청크")

    print(f"\n총 청크: {len(docs)}개 — TF-IDF 벡터화 중...")
    vectorizer   = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
    matrix       = vectorizer.fit_transform([d['text'] for d in docs])
    matrix_norm  = normalize(matrix, norm='l2')

    pickle.dump({'vectorizer': vectorizer, 'matrix': matrix_norm, 'docs': docs},
                open(INDEX_PATH, 'wb'))
    print(f"\n✅ 인덱스 저장 완료 → {INDEX_PATH}")
    print(f"   shape: {matrix_norm.shape}")


In [ ]:
# ── 셀 1-2: 인덱스 로드 ──────────────────────────────────────
_idx = pickle.load(open(os.path.join(BASE_DIR, 'tfidf_index.pkl'), 'rb'))
VECTORIZER = _idx['vectorizer']
MATRIX     = _idx['matrix']
DOCS       = _idx['docs']
print(f"✅ 인덱스 로드 완료 — 청크 {len(DOCS)}개 / 어휘 {len(VECTORIZER.vocabulary_)}개")

---
## 2. Few-shot 풀 & QA 캐시 구축
TL 서술형(q_type=3) → `fewshot_pool.pkl`  
TL/VL 전체 → `qa_cache.pkl` (LLM-as-Judge용)

> 최초 1회만 실행.

In [ ]:
# ── 셀 2-1: 압축 해제 + Few-shot 풀 & QA 캐시 구축 (최초 1회) ──
FEWSHOT_PATH = os.path.join(BASE_DIR, 'fewshot_pool.pkl')
QA_PATH      = os.path.join(BASE_DIR, 'qa_cache.pkl')

if os.path.exists(FEWSHOT_PATH) and os.path.exists(QA_PATH):
    print("ℹ️  pkl 파일 이미 존재. 다음 셀(2-2)로 건너뛰세요.")
else:
    import zipfile, glob

    # ── Step 1: TL/VL 압축 해제 ────────────────────────────────
    for prefix, src_dir, extract_base in [
        ('TL', TL_DIR, os.path.join(BASE_DIR, 'tl_data')),
        ('VL', VL_DIR, os.path.join(BASE_DIR, 'vl_data')),
    ]:
        os.makedirs(extract_base, exist_ok=True)
        zips = glob.glob(os.path.join(src_dir, f'{prefix}_*.zip.part0'))
        for zip_path in zips:
            name    = os.path.basename(zip_path).replace('.zip.part0', '')
            out_dir = os.path.join(extract_base, name.replace(f'{prefix}_', ''))
            if os.path.isdir(out_dir) and len(glob.glob(f'{out_dir}/**/*.json', recursive=True)) > 0:
                print(f"  ⏭️  {name} 이미 해제됨")
                continue
            os.makedirs(out_dir, exist_ok=True)
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(out_dir)
                count = len(glob.glob(f'{out_dir}/**/*.json', recursive=True))
                print(f"  ✅ {name}: {count}개 JSON")
            except Exception as e:
                print(f"  ❌ {name}: {e}")

    # ── Step 2: Few-shot 풀 & QA 캐시 구축 ─────────────────────
    print("\nFew-shot 풀 & QA 캐시 구축 중...")
    pool   = defaultdict(list)
    tl_all, vl_all = [], []

    for fp in glob.glob(os.path.join(BASE_DIR, 'tl_data', '**', '*.json'), recursive=True):
        with open(fp, 'r', encoding='utf-8-sig') as f:
            try:
                d = json.load(f)
                tl_all.append(d)
                if d.get('q_type') == 3:
                    dom = d.get('domain', 13)
                    pool[dom].append({'question': d['question'], 'answer': d['answer'],
                                      'dept': DOMAIN_MAP.get(dom, '기타')})
            except:
                pass

    for fp in glob.glob(os.path.join(BASE_DIR, 'vl_data', '**', '*.json'), recursive=True):
        with open(fp, 'r', encoding='utf-8-sig') as f:
            try:
                vl_all.append(json.load(f))
            except:
                pass

    total = sum(len(v) for v in pool.values())
    print(f"서술형 QA {total}건 / TL {len(tl_all)}건 / VL {len(vl_all)}건")

    pickle.dump(dict(pool), open(FEWSHOT_PATH, 'wb'))
    pickle.dump({'tl': tl_all, 'vl': vl_all}, open(QA_PATH, 'wb'))
    print(f"\n✅ fewshot_pool.pkl / qa_cache.pkl 저장 완료 → {BASE_DIR}")


In [ ]:
# ── 셀 2-2: 로드 ─────────────────────────────────────────────
FEWSHOT = pickle.load(open(os.path.join(BASE_DIR, 'fewshot_pool.pkl'), 'rb'))
QA_DATA = pickle.load(open(os.path.join(BASE_DIR, 'qa_cache.pkl'), 'rb'))
print(f"✅ Few-shot 풀: {sum(len(v) for v in FEWSHOT.values())}건")
print(f"✅ QA 캐시: TL {len(QA_DATA['tl'])}건 / VL {len(QA_DATA['vl'])}건")

---
## 3. 권역별 병원 DB

In [ ]:
REGIONS = {
    '수도권':     ['서울','경기','인천'],
    '충청권':     ['대전','충남','충북','세종'],
    '호남권':     ['광주','전남','전북'],
    '대구경북권': ['대구','경북'],
    '부산경남권': ['부산','울산','경남'],
    '강원권':     ['강원'],
    '제주권':     ['제주'],
}

# Track A — 1·2차 지역 병원
TRACK_A_DB = {
    '수도권': [
        {'name':'서울 열린내과의원',  'addr':'서울 강남구', 'depts':['내과','외과','신경과'],        'type':'의원', 'rating':4.5},
        {'name':'경기중앙병원',        'addr':'경기 수원시', 'depts':['정형외과','내과','피부과'],    'type':'병원', 'rating':4.3},
        {'name':'인천현대의원',        'addr':'인천 남동구', 'depts':['내과','이비인후과','안과'],    'type':'의원', 'rating':4.2},
        {'name':'강남연세병원',        'addr':'서울 서초구', 'depts':['외과','정형외과','비뇨의학과'],'type':'병원', 'rating':4.4},
    ],
    '충청권': [
        {'name':'대전중앙내과',   'addr':'대전 서구',   'depts':['내과','신경과','정신건강의학과'],'type':'의원','rating':4.3},
        {'name':'충청종합병원',   'addr':'충남 천안시', 'depts':['외과','내과','이비인후과'],      'type':'병원','rating':4.1},
        {'name':'세종으뜸의원',   'addr':'세종시',      'depts':['내과','피부과','안과'],          'type':'의원','rating':4.2},
    ],
    '호남권': [
        {'name':'광주제일의원',   'addr':'광주 북구',   'depts':['내과','외과','이비인후과'],      'type':'의원','rating':4.2},
        {'name':'전남중앙병원',   'addr':'전남 목포시', 'depts':['신경과','내과','정형외과'],      'type':'병원','rating':4.0},
        {'name':'전북현대의원',   'addr':'전북 전주시', 'depts':['내과','피부과','비뇨의학과'],    'type':'의원','rating':4.3},
    ],
    '대구경북권': [
        {'name':'대구중앙의원',   'addr':'대구 중구',   'depts':['내과','외과','이비인후과'],      'type':'의원','rating':4.4},
        {'name':'경북종합병원',   'addr':'경북 포항시', 'depts':['정형외과','신경과','내과'],      'type':'병원','rating':4.1},
    ],
    '부산경남권': [
        {'name':'부산해운대의원', 'addr':'부산 해운대구','depts':['내과','피부과','이비인후과'],   'type':'의원','rating':4.3},
        {'name':'경남중앙병원',   'addr':'경남 창원시', 'depts':['외과','신경과','비뇨의학과'],    'type':'병원','rating':4.2},
        {'name':'울산현대의원',   'addr':'울산 남구',   'depts':['내과','정형외과','안과'],        'type':'의원','rating':4.1},
    ],
    '강원권': [
        {'name':'강원중앙의원',          'addr':'강원 춘천시', 'depts':['내과','외과','이비인후과'],'type':'의원','rating':4.2},
        {'name':'원주세브란스병원분원',   'addr':'강원 원주시', 'depts':['신경과','내과','외과'],   'type':'병원','rating':4.4},
    ],
    '제주권': [
        {'name':'제주중앙의원', 'addr':'제주시',   'depts':['내과','피부과','외과'],  'type':'의원','rating':4.3},
        {'name':'서귀포의료원','addr':'서귀포시', 'depts':['내과','외과','신경과'],   'type':'병원','rating':4.1},
    ],
}

# Track B — 상급종합 3차 병원
TRACK_B_DB = {
    '수도권': [
        {'name':'서울대학교병원',  'addr':'서울 종로구',  'specialty':['신경외과','심장외과','종양내과','혈액종양과'],    'level':'상급종합','prof_count':320},
        {'name':'세브란스병원',    'addr':'서울 서대문구','specialty':['신경과','소화기내과','심장내과','내분비내과'],    'level':'상급종합','prof_count':280},
        {'name':'삼성서울병원',    'addr':'서울 강남구',  'specialty':['암센터','뇌신경센터','심혈관센터','희귀질환센터'],'level':'상급종합','prof_count':300},
        {'name':'서울아산병원',    'addr':'서울 송파구',  'specialty':['간이식','심장이식','신경외과','소아청소년과'],    'level':'상급종합','prof_count':310},
    ],
    '충청권': [
        {'name':'충남대학교병원', 'addr':'대전 중구', 'specialty':['신경외과','혈액종양과','심장내과'],   'level':'상급종합','prof_count':120},
        {'name':'건양대학교병원', 'addr':'대전 서구', 'specialty':['안과','이비인후과','내과'],           'level':'상급종합','prof_count':95},
    ],
    '호남권': [
        {'name':'전남대학교병원', 'addr':'광주 동구', 'specialty':['혈액종양과','신경과','심장내과'],     'level':'상급종합','prof_count':140},
        {'name':'조선대학교병원', 'addr':'광주 동구', 'specialty':['외과','이비인후과','비뇨의학과'],     'level':'상급종합','prof_count':110},
    ],
    '대구경북권': [
        {'name':'경북대학교병원',       'addr':'대구 중구',   'specialty':['신경외과','혈액종양과','소화기내과'],'level':'상급종합','prof_count':150},
        {'name':'계명대학교동산병원',   'addr':'대구 달서구', 'specialty':['심장내과','내분비내과','신경과'],   'level':'상급종합','prof_count':130},
    ],
    '부산경남권': [
        {'name':'부산대학교병원',       'addr':'부산 서구',   'specialty':['신경외과','혈액종양과','심장외과'],  'level':'상급종합','prof_count':160},
        {'name':'양산부산대학교병원',   'addr':'경남 양산시', 'specialty':['소화기내과','내분비내과','신경과'],  'level':'상급종합','prof_count':140},
    ],
    '강원권': [
        {'name':'강원대학교병원',         'addr':'강원 춘천시', 'specialty':['신경과','외과','내과'],             'level':'상급종합','prof_count':80},
        {'name':'한림대학교춘천성심병원', 'addr':'강원 춘천시', 'specialty':['심장내과','신경외과','비뇨의학과'],  'level':'상급종합','prof_count':75},
    ],
    '제주권': [
        {'name':'제주대학교병원', 'addr':'제주시', 'specialty':['신경과','내과','외과','혈액종양과'], 'level':'상급종합','prof_count':60},
    ],
}

def get_region(sido: str) -> str:
    for region, sidos in REGIONS.items():
        if any(s in sido for s in sidos):
            return region
    return '수도권'

def query_track_a(region: str, dept_keywords: list) -> list:
    hospitals = TRACK_A_DB.get(region, TRACK_A_DB['수도권'])
    scored = sorted(hospitals,
        key=lambda h: (-sum(1 for kw in dept_keywords if any(kw in d for d in h['depts'])),
                       -h['rating']))
    return scored[:3]

def query_track_b(region: str, disease_keywords: list) -> list:
    hospitals = TRACK_B_DB.get(region, TRACK_B_DB['수도권'])
    scored = sorted(hospitals,
        key=lambda h: (-sum(1 for kw in disease_keywords if any(kw in s for s in h['specialty'])),
                       -h['prof_count']))
    return scored[:3]

print("✅ 병원 DB 로드 완료 — Track A/B 각 7개 권역")

---
## 4. 파이프라인 함수 정의

In [ ]:
# ── Step 2-A: RAG Retrieval ───────────────────────────────────
def retrieve(query: str, top_k: int = 5) -> list:
    """TF-IDF 기반 관련 문서 검색 — SentenceTransformer로 교체 가능"""
    qvec = VECTORIZER.transform([query])
    qvec = normalize(qvec, norm='l2')
    scores = (MATRIX @ qvec.T).toarray().flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    return [{'text': DOCS[i]['text'], 'score': float(scores[i]),
             'cat': DOCS[i]['cat']} for i in top_idx]

# ── Few-shot 예시 선택 ────────────────────────────────────────
def get_fewshot(domain_id: int, n: int = 2) -> str:
    pool = FEWSHOT.get(domain_id, FEWSHOT.get(13, []))
    samples = random.sample(pool, min(n, len(pool)))
    return "\n\n".join(
        f"[전문 소견]\n{s['question'][:200]}\n[쉬운 말 설명]\n{s['answer'][:300]}"
        for s in samples
    )

# ── Step 2-B: 쉬운 말 설명 생성 ──────────────────────────────
def generate_easy_explanation(input_text: str, retrieved_docs: list, domain_id: int) -> str:
    context  = "\n---\n".join([d['text'] for d in retrieved_docs[:3]])
    fewshot  = get_fewshot(domain_id)
    prompt = f"""당신은 환자에게 의료 정보를 쉽게 설명해주는 전문가입니다.
아래 참고 의학 지식과 예시를 바탕으로, 환자가 이해하기 쉬운 설명을 작성하세요.
초등학생도 이해할 수 있는 일상 언어를 사용하되, 의학적 사실은 반드시 보존하세요.

[참고 의학 지식]
{context}

[변환 예시]
{fewshot}

[환자 소견/진단]
{input_text}

[쉬운 말 설명] (200~300자, 친절하고 명확하게):"""

    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=500,
        messages=[{"role":"user","content":prompt}]
    )
    return resp.content[0].text.strip()

# ── Step 2-C: 진료과·키워드·중증도 추출 ──────────────────────
def extract_dept_and_keywords(input_text: str) -> dict:
    dept_list = list(DOMAIN_MAP.values())
    prompt = f"""다음 의료 소견에서 정보를 추출하세요. 반드시 JSON만 출력하세요.

소견: {input_text}

출력 형식:
{{
  "dept": "진료과명 (다음 중 하나: {', '.join(dept_list)})",
  "severity": "경증 또는 중증",
  "urgency": "일반 또는 긴급",
  "keywords": ["질병키워드1", "질병키워드2", "질병키워드3"],
  "reason": "중증도 판단 근거 한 줄"
}}"""

    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=300,
        messages=[{"role":"user","content":prompt}]
    )
    text = resp.content[0].text.strip()
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    return {"dept":"기타","severity":"경증","urgency":"일반","keywords":[],"reason":"분석 실패"}

# ── Step 5: 최종 리포트 합성 ──────────────────────────────────
def synthesize_report(input_text: str, easy_explanation: str,
                      triage: dict, hospitals: list, track: str) -> str:
    hosp_lines = []
    for i, h in enumerate(hospitals, 1):
        if track == 'A':
            hosp_lines.append(f"{i}. {h['name']} ({h['addr']}) — {h['type']} | "
                              f"주요과: {', '.join(h['depts'])} | 평점: {h['rating']}")
        else:
            hosp_lines.append(f"{i}. {h['name']} ({h['addr']}) — {h['level']} | "
                              f"전문분야: {', '.join(h['specialty'])} | 전문의 {h['prof_count']}명")

    referral = ("\n⚠️ 상급종합병원 방문 시 1차 의원에서 **진료의뢰서(소견서)**를 "
                "먼저 발급받으시면 진료비 혜택을 받을 수 있습니다.\n") if track == 'B' else ""

    prompt = f"""다음 정보를 바탕으로 환자에게 전달할 최종 안내 리포트를 작성하세요.
친절하고 명확하게, 불필요한 의학 용어는 배제하세요.

[원본 소견] {input_text}
[쉬운 말 설명] {easy_explanation}
[진료 트랙] {'Track B — 상급종합 3차 병원' if track=='B' else 'Track A — 지역 1·2차 병원'}
[판단 근거] {triage.get('reason','')}
[추천 의료기관]\n{'  '.join(hosp_lines)}
{referral}

아래 구조로 작성하세요:
1. 📋 현재 상태 요약 (쉬운 말, 3~4문장)
2. 🏥 추천 의료기관 목록 (기관명, 추천 이유 포함)
3. ⚠️ 주의사항 및 면책 고지"""

    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=800,
        messages=[{"role":"user","content":prompt}]
    )
    return resp.content[0].text.strip()

# ── Step 6: LLM-as-Judge ─────────────────────────────────────
def llm_judge(original_text: str, generated_report: str) -> dict:
    prompt = f"""당신은 의료 AI 출력물의 품질을 평가하는 전문 판정관입니다.

[원본 환자 소견] {original_text}
[AI 생성 리포트] {generated_report}

다음 루브릭으로 평가하고 반드시 JSON만 출력하세요:
{{
  "omission":   {{"score": 0~2, "comment": "평가 근거"}},
  "distortion": {{"score": 0~2, "comment": "평가 근거"}},
  "safety":     {{"score": 0~2, "comment": "평가 근거"}},
  "total": 0~6,
  "verdict": "PASS 또는 FAIL",
  "feedback": "개선 필요사항 한 줄"
}}
  - omission:   0=심각한 누락, 1=경미한 누락, 2=완전
  - distortion: 0=심각한 왜곡, 1=경미, 2=정확
  - safety:     0=면책문구 없음, 1=부분, 2=완전"""

    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=400,
        messages=[{"role":"user","content":prompt}]
    )
    m = re.search(r'\{.*\}', resp.content[0].text.strip(), re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    return {"total":0,"verdict":"FAIL","feedback":"판정 실패"}

print("✅ 파이프라인 함수 정의 완료")

---
## 5. 단일 케이스 테스트
Gradio 실행 전 파이프라인이 정상 작동하는지 확인

In [ ]:
# 테스트 입력
TEST_INPUT = "흉부 X-ray 판독 결과 우측 하엽에 Pleural Effusion 소견이 관찰되며 Cardiomegaly 동반. 심비대 및 흉수 증가 추이 확인 필요."
TEST_SIDO  = "서울"

print("=" * 60)
print("[입력]")
print(TEST_INPUT)
print("[지역]", TEST_SIDO)
print("=" * 60)

# Step 2-A: RAG
print("\n▶ Step 2-A: RAG 검색")
retrieved = retrieve(TEST_INPUT, top_k=5)
for d in retrieved:
    print(f"  score={d['score']:.3f} | {d['cat']} | {d['text'][:60]}...")

# Step 2-C: 진료과/중증도 추출
print("\n▶ Step 2-C: 진료과·중증도 추출")
triage = extract_dept_and_keywords(TEST_INPUT)
print(json.dumps(triage, ensure_ascii=False, indent=2))

# Step 2-B: 쉬운 말 생성
print("\n▶ Step 2-B: 쉬운 말 설명 생성")
dept      = triage.get('dept', '기타')
domain_id = next((k for k, v in DOMAIN_MAP.items() if v == dept), 13)
easy      = generate_easy_explanation(TEST_INPUT, retrieved, domain_id)
print(easy)

# Step 3: 트랙 분기
track = 'B' if triage.get('severity') == '중증' or triage.get('urgency') == '긴급' else 'A'
print(f"\n▶ Step 3: {'Track B (상급종합)' if track=='B' else 'Track A (지역병원)'}")

# Step 4: 병원 매칭
region    = get_region(TEST_SIDO)
keywords  = triage.get('keywords', [])
hospitals = query_track_b(region, keywords) if track == 'B' else query_track_a(region, [dept] + keywords)
print(f"▶ Step 4: {region} 권역 병원 매칭")
for h in hospitals:
    print(f"  {h['name']} | {h['addr']}")

# Step 5: 리포트 합성
print("\n▶ Step 5: 최종 리포트 합성")
report = synthesize_report(TEST_INPUT, easy, triage, hospitals, track)
print(report)

# Step 6: LLM-as-Judge
print("\n▶ Step 6: LLM-as-Judge")
judge = llm_judge(TEST_INPUT, report)
print(json.dumps(judge, ensure_ascii=False, indent=2))

---
## 6. Gradio UI 실행

In [ ]:
import gradio as gr

SIDO_LIST = [f"{s} ({r})" for r, sidos in REGIONS.items() for s in sidos]

SAMPLE_INPUTS = [
    "흉부 X-ray 판독 결과 우측 하엽에 Pleural Effusion 소견이 관찰되며 Cardiomegaly 동반. 심비대 및 흉수 증가 추이 확인 필요.",
    "MRI 결과 L4-L5 추간판 탈출증(Herniated Nucleus Pulposus) 확인. 우측 하지 방사통 및 감각 저하 동반.",
    "혈액 검사상 HbA1c 9.2%, 공복혈당 210mg/dL. 제2형 당뇨 조절 불량 소견. 합병증 정밀 평가 필요.",
    "갑작스러운 두통과 좌측 상하지 위약감. CT상 우측 기저핵 부위 출혈 의심 소견.",
]

def run_pipeline(input_text: str, sido: str, progress=gr.Progress()):
    if not input_text.strip():
        return ("입력 내용이 없습니다.", "", "", "", "", "")

    logs = []
    sido_name = sido.split(' (')[0]
    region    = get_region(sido_name)

    progress(0.10, desc="의학 지식 검색 중...")
    retrieved = retrieve(input_text, top_k=5)
    logs.append(f"✅ RAG 검색 완료 — 관련 문서 {len(retrieved)}건")

    progress(0.25, desc="진료과 및 중증도 분석 중...")
    triage    = extract_dept_and_keywords(input_text)
    dept      = triage.get('dept', '기타')
    severity  = triage.get('severity', '경증')
    urgency   = triage.get('urgency', '일반')
    keywords  = triage.get('keywords', [])
    domain_id = next((k for k, v in DOMAIN_MAP.items() if v == dept), 13)
    logs.append(f"✅ 진료과: {dept} | 중증도: {severity} | 긴급도: {urgency}")
    logs.append(f"✅ 질병 키워드: {', '.join(keywords)}")

    progress(0.40, desc="쉬운 말 설명 생성 중...")
    easy = generate_easy_explanation(input_text, retrieved, domain_id)
    logs.append("✅ 쉬운 말 설명 생성 완료")

    track = 'B' if severity == '중증' or urgency == '긴급' else 'A'
    logs.append(f"✅ {'Track B — 상급종합 3차' if track=='B' else 'Track A — 지역 1·2차'} 분기")

    progress(0.60, desc="병원 매칭 중...")
    hospitals = (query_track_b(region, keywords) if track == 'B'
                 else query_track_a(region, [dept] + keywords))
    logs.append(f"✅ {region} 권역 병원 {len(hospitals)}곳 매칭")

    progress(0.75, desc="최종 리포트 생성 중...")
    report = synthesize_report(input_text, easy, triage, hospitals, track)
    logs.append("✅ 최종 리포트 생성 완료")

    progress(0.90, desc="품질 검증 중...")
    judge   = llm_judge(input_text, report)
    verdict = judge.get('verdict', 'FAIL')
    total   = judge.get('total', 0)
    logs.append(f"✅ 검증 완료 — {verdict} ({total}/6점)")

    j = judge
    judge_md = f"""### 판정 결과: {'✅ PASS' if verdict=='PASS' else '❌ FAIL'} ({total}/6점)

| 항목 | 점수 | 평가 |
|---|---|---|
| 핵심 정보 보존  | {j.get('omission',{}).get('score','?')}/2 | {j.get('omission',{}).get('comment','—')} |
| 수치·심각도 정확성 | {j.get('distortion',{}).get('score','?')}/2 | {j.get('distortion',{}).get('comment','—')} |
| 안전 문구 포함  | {j.get('safety',{}).get('score','?')}/2 | {j.get('safety',{}).get('comment','—')} |

**개선 의견:** {j.get('feedback','')}"""

    rag_md = "### 참고 의학 문서 (Top 5)\n\n" + "\n\n".join(
        f"**[{i+1}]** `{d['cat']}` (유사도 {d['score']:.3f})\n\n{d['text'][:200]}..."
        for i, d in enumerate(retrieved)
    )

    triage_md = f"""### 분석 결과
- **진료과:** {dept}
- **중증도:** {severity} | **긴급도:** {urgency}
- **질병 키워드:** {', '.join(keywords)}
- **판단 근거:** {triage.get('reason','')}
- **배정 트랙:** {'Track B — 상급종합 3차' if track=='B' else 'Track A — 지역 1·2차'}
- **권역:** {region}"""

    return easy, triage_md, report, judge_md, rag_md, "\n".join(logs)

# ── UI 레이아웃 ───────────────────────────────────────────────
with gr.Blocks(
    title="환자 맞춤형 의료 안내 서비스",
    theme=gr.themes.Soft(primary_hue="teal", secondary_hue="slate")
) as demo:

    gr.HTML("""
    <div style='text-align:center; padding:20px 0 10px'>
      <h1>🏥 환자 맞춤형 의료 안내 서비스</h1>
      <p>소견서·진단 텍스트를 입력하면 쉬운 말 설명과 맞춤형 병원을 안내해드립니다.</p>
    </div>
    <div style='font-size:12px;color:#888;border:1px solid #ddd;padding:10px;border-radius:6px'>
      ⚠️ 본 서비스는 의료 정보 제공 목적의 AI 보조 도구입니다.
      진단·처방을 대체하지 않으며, 최종 의료 판단은 반드시 전문의에게 받으시기 바랍니다.
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            inp_text = gr.Textbox(
                label="소견서 / 진단 텍스트",
                placeholder="의사 소견서, 진단서, 검사 결과 등 텍스트를 입력하세요...",
                lines=6
            )
            inp_sido = gr.Dropdown(
                choices=SIDO_LIST, value="서울 (수도권)",
                label="거주 지역 (시·도)"
            )
            with gr.Row():
                btn_run   = gr.Button("🔍 분석 시작", variant="primary", scale=3)
                btn_clear = gr.Button("🗑️ 초기화", scale=1)

            gr.Examples(
                examples=[[s, "서울 (수도권)"] for s in SAMPLE_INPUTS],
                inputs=[inp_text, inp_sido],
                label="예시 입력"
            )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("💬 쉬운 말 설명"):
                    out_easy   = gr.Markdown()
                with gr.Tab("🔍 중증도 분석"):
                    out_triage = gr.Markdown()
                with gr.Tab("🏥 최종 리포트"):
                    out_report = gr.Markdown()
                with gr.Tab("⚖️ LLM-as-Judge"):
                    out_judge  = gr.Markdown()
                with gr.Tab("📚 RAG 근거 문서"):
                    out_rag    = gr.Markdown()
                with gr.Tab("📋 처리 로그"):
                    out_log    = gr.Markdown()

    btn_run.click(
        fn=run_pipeline,
        inputs=[inp_text, inp_sido],
        outputs=[out_easy, out_triage, out_report, out_judge, out_rag, out_log]
    )
    btn_clear.click(
        fn=lambda: ("", "서울 (수도권)"),
        outputs=[inp_text, inp_sido]
    )

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)